In [0]:
from pyspark.sql import Row

master_data = [
    Row(customer_id=101, name="Alice", city="New York", email="alice@email.com"),
    Row(customer_id=102, name="Bob", city="Chicago", email="bob@email.com"),
    Row(customer_id=103, name="Charlie", city="Dallas", email="charlie@email.com"),
    Row(customer_id=104, name="David", city="Miami", email="david@email.com"),
    Row(customer_id=104, name="David", city="Miami", email="david@email.com"),  # Duplicate
    Row(customer_id=105, name=None, city="Boston", email="emma@email.com")       # Null Value
]

master_df = spark.createDataFrame(master_data)

display(master_df)

customer_id,name,city,email
101,Alice,New York,alice@email.com
102,Bob,Chicago,bob@email.com
103,Charlie,Dallas,charlie@email.com
104,David,Miami,david@email.com
104,David,Miami,david@email.com
105,null,Boston,emma@email.com


In [0]:
# Remove duplicate rows
clean_df = master_df.dropDuplicates()

# Replace null values in the 'name' column
clean_df = clean_df.fillna({"name": "Unknown"})

display(clean_df)

customer_id,name,city,email
101,Alice,New York,alice@email.com
102,Bob,Chicago,bob@email.com
103,Charlie,Dallas,charlie@email.com
104,David,Miami,david@email.com
105,Unknown,Boston,emma@email.com


In [0]:
clean_df.write.format("delta").mode("overwrite").saveAsTable("customer_master")

In [0]:
# Read the Delta table
delta_df = spark.read.table("customer_master")

display(delta_df)

customer_id,name,city,email
101,Alice,New York,alice@email.com
102,Bob,Chicago,bob@email.com
103,Charlie,Dallas,charlie@email.com
104,David,Miami,david@email.com
105,Unknown,Boston,emma@email.com


In [0]:
from pyspark.sql import Row

incremental_data = [
    Row(customer_id=102, name="Bob", city="Seattle", email="bob@email.com"),      # Update existing customer
    Row(customer_id=106, name="Frank", city="Denver", email="frank@email.com"),    # New customer
    Row(customer_id=107, name="Grace", city="Austin", email="grace@email.com")     # New customer
]

incremental_df = spark.createDataFrame(incremental_data)

display(incremental_df)

customer_id,name,city,email
102,Bob,Seattle,bob@email.com
106,Frank,Denver,frank@email.com
107,Grace,Austin,grace@email.com


In [0]:
from delta.tables import DeltaTable

delta_table = DeltaTable.forName(spark, "customer_master")

(
    delta_table.alias("target")
    .merge(
        incremental_df.alias("source"),
        "target.customer_id = source.customer_id"
    )
    .whenMatchedUpdate(set={
        "name": "source.name",
        "city": "source.city",
        "email": "source.email"
    })
    .whenNotMatchedInsert(values={
        "customer_id": "source.customer_id",
        "name": "source.name",
        "city": "source.city",
        "email": "source.email"
    })
    .execute()
)

print("MERGE completed successfully!")

MERGE completed successfully!


In [0]:
# Read the updated Delta table
final_df = spark.read.table("customer_master")

# Display final data
display(final_df)

# Row count
print("Total Rows:", final_df.count())

# Check for duplicate customer IDs
duplicate_count = (
    final_df.groupBy("customer_id")
            .count()
            .filter("count > 1")
)

print("Duplicate Records:")
display(duplicate_count)

customer_id,name,city,email
101,Alice,New York,alice@email.com
103,Charlie,Dallas,charlie@email.com
104,David,Miami,david@email.com
105,Unknown,Boston,emma@email.com
102,Bob,Seattle,bob@email.com
106,Frank,Denver,frank@email.com
107,Grace,Austin,grace@email.com


Total Rows: 7
Duplicate Records:


customer_id,count


In [0]:
display(
    final_df.orderBy("customer_id")
)

customer_id,name,city,email
101,Alice,New York,alice@email.com
102,Bob,Seattle,bob@email.com
103,Charlie,Dallas,charlie@email.com
104,David,Miami,david@email.com
105,Unknown,Boston,emma@email.com
106,Frank,Denver,frank@email.com
107,Grace,Austin,grace@email.com
